# DB 데이터 추출 및 CSV 저장

PostgreSQL DB에서 수집된 Steam 인디게임 데이터를 불러와 `data/raw/`에 CSV로 저장한다.

| 테이블 | 설명 | 행 수 | 저장 파일 |
|---|---|---|---|
| `steam_indie_games` | 게임 메타데이터 (장르·가격·출시일 등) | 9,593 | `steam_indie_games.csv` |
| `steam_indie_tags` | SteamSpy 태그 데이터 | 9,706 | `steam_indie_tags.csv` |
| `steam_indie_reviews` | 리뷰 원문 및 작성자 정보 | 236,379 | `steam_indie_reviews.csv` |
| `steam_indie_review_histogram` | 월별/일별 리뷰 집계 | 11,782 | `steam_indie_review_histogram.csv` |
| `steam_indie_review_summary` | 게임별 리뷰 요약 통계 | 200 | `steam_indie_review_summary.csv` |

## 라이브러리 임포트 및 DB 연결

In [8]:
import sys
import json as _json
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve().parents[1] / 'src'))
from utils.db import get_connection

PROCESSED_DIR = Path('..').resolve().parents[1] / 'data' / 'raw'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

conn = get_connection()
print('DB 연결 성공')
print(f'저장 경로: {PROCESSED_DIR}')

DB 연결 성공
저장 경로: /Users/jin/Develop/codingclub/game-analysis/data/raw


## 1. steam_indie_games

Steam Store API로 수집한 게임 메타데이터. 장르, 가격, 출시일, 소유자 수 등 핵심 정보를 포함한다.

In [9]:
df_games = pd.read_sql('SELECT * FROM steam_indie_games ORDER BY appid', conn)

out = PROCESSED_DIR / 'steam_indie_games.csv'
df_games.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_games: {len(df_games):,}행 → {out.name}')
print(f'컬럼: {df_games.columns.tolist()}')
df_games.head(3)

/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_10313/2578615608.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_games = pd.read_sql('SELECT * FROM steam_indie_games ORDER BY appid', conn)


steam_indie_games: 9,692행 → steam_indie_games.csv
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'total_reviews', 'owners_lower', 'is_f2p', 'is_early_access', 'name']


,appid,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name
0,226620,"200,000 .. 500,000",1912,364,1499,12,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",2023-04-18,QCF Design,2276,200000,False,False,Desktop Dungeons
1,230210,"0 .. 20,000",303,45,2499,3,"['Adventure', 'Indie']",2025-03-13,Senscape,348,0,False,False,ASYLUM
2,251570,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False,7 Days to Die


## 2. steam_indie_tags

SteamSpy API로 수집한 게임 태그 데이터. `tags` 컬럼은 JSONB 형식으로 저장되어 있어 문자열로 변환한다.

In [10]:
df_tags = pd.read_sql('SELECT * FROM steam_indie_tags ORDER BY appid', conn)

# tags 컬럼(JSONB) → 문자열 변환
df_tags['tags'] = df_tags['tags'].apply(
    lambda x: _json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else x
)

out = PROCESSED_DIR / 'steam_indie_tags.csv'
df_tags.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_tags: {len(df_tags):,}행 → {out.name}')
print(f'컬럼: {df_tags.columns.tolist()}')
df_tags.head(3)

/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_10313/1584207267.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tags = pd.read_sql('SELECT * FROM steam_indie_tags ORDER BY appid', conn)


steam_indie_tags: 9,706행 → steam_indie_tags.csv
컬럼: ['appid', 'name', 'developer', 'publisher', 'owners', 'positive', 'negative', 'price', 'tags', 'updated_at']


,appid,name,developer,publisher,owners,positive,negative,price,tags,updated_at
0,226620,Desktop Dungeons,QCF Design,QCF Design,"200,000 .. 500,000",1912,364,1499,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ...",2026-04-27 20:02:19.802355
1,230210,ASYLUM,Senscape,Senscape,"0 .. 20,000",303,45,2499,"{""Dark"": 40, ""Gore"": 52, ""Indie"": 76, ""Gothic""...",2026-04-27 21:13:17.714243
2,251570,7 Days to Die,The Fun Pimps,The Fun Pimps Entertainment LLC,"10,000,000 .. 20,000,000",327889,42157,4499,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""...",2026-04-27 19:17:36.995662


## 3. steam_indie_reviews

수집된 전체 리뷰 데이터. 236,000건 이상으로 용량이 크므로 청크 단위로 읽어 저장한다.

In [11]:
out = PROCESSED_DIR / 'steam_indie_reviews.csv'

chunk_size = 50000
total = 0
for i, chunk in enumerate(
    pd.read_sql('SELECT * FROM steam_indie_reviews ORDER BY appid, timestamp_created', conn, chunksize=chunk_size)
):
    chunk.to_csv(out, index=False, encoding='utf-8-sig', mode='w' if i == 0 else 'a', header=(i == 0))
    total += len(chunk)
    print(f'  청크 {i+1}: {total:,}행 저장 완료')

print(f'\nsteam_indie_reviews: 총 {total:,}행 → {out.name}')

/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_10313/204090164.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM steam_indie_reviews ORDER BY appid, timestamp_created', conn, chunksize=chunk_size)


  청크 1: 50,000행 저장 완료
  청크 2: 100,000행 저장 완료
  청크 3: 150,000행 저장 완료
  청크 4: 200,000행 저장 완료
  청크 5: 236,379행 저장 완료

steam_indie_reviews: 총 236,379행 → steam_indie_reviews.csv


## 4. steam_indie_review_histogram

게임별 월별(`rollups`) 및 일별(`recent`) 리뷰 집계 데이터.

In [12]:
df_hist = pd.read_sql(
    'SELECT * FROM steam_indie_review_histogram ORDER BY appid, data_type, date', conn
)

out = PROCESSED_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_histogram: {len(df_hist):,}행 → {out.name}')
print(f'컬럼: {df_hist.columns.tolist()}')
print(f'\ndata_type 분포:')
print(df_hist['data_type'].value_counts().to_string())
df_hist.head(3)

/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_10313/3025296071.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hist = pd.read_sql(


steam_indie_review_histogram: 11,782행 → steam_indie_review_histogram.csv
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']

data_type 분포:
data_type
recent     5976
rollups    5806


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent


## 5. steam_indie_review_summary

게임별 리뷰 요약 통계 (review_score, 긍정/부정 수 등). 수집 시점의 전체 누적 리뷰 기준이다.

In [13]:
df_summary = pd.read_sql('SELECT * FROM steam_indie_review_summary ORDER BY appid', conn)

out = PROCESSED_DIR / 'steam_indie_review_summary.csv'
df_summary.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_summary: {len(df_summary):,}행 → {out.name}')
print(f'컬럼: {df_summary.columns.tolist()}')
df_summary.head(3)

steam_indie_review_summary: 200행 → steam_indie_review_summary.csv
컬럼: ['appid', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews', 'collected_at']


/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_10313/1754953098.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_summary = pd.read_sql('SELECT * FROM steam_indie_review_summary ORDER BY appid', conn)


,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews,collected_at
0,402160,5,Mixed,152,205,357,1777436594
1,437440,5,Mixed,66,30,96,1777439411
2,444690,5,Mixed,111,135,246,1777436587


## 6. DB 연결 종료 및 저장 결과 요약

In [14]:
conn.close()
print('DB 연결 종료')

print('\n=== 저장 완료 파일 목록 ===')
for f in sorted(PROCESSED_DIR.glob('steam_indie_*.csv')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<50} {size_mb:>7.1f} MB')

DB 연결 종료

=== 저장 완료 파일 목록 ===
  steam_indie_games.csv                                  1.2 MB
  steam_indie_review_histogram.csv                       1.1 MB
  steam_indie_review_summary.csv                         0.0 MB
  steam_indie_reviews.csv                               76.6 MB
  steam_indie_tags.csv                                   4.4 MB
